<a href="https://colab.research.google.com/github/ronan-cunha/machine-learning-course/blob/main/Notebooks/aula_8_avaliacao_modelos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [29]:
# Carrega o dataset Fair (sobre casos extraconjugais)
# https://www.statsmodels.org/stable/datasets/generated/fair.html
affair = sm.datasets.fair.load_pandas()
df_real = affair.data
display(df_real.head())

,rate_marriage,age,yrs_married,children,religious,educ,occupation,occupation_husb,affairs
0,3.0,32.0,9.0,3.0,3.0,17.0,2.0,5.0,0.111111
1,3.0,27.0,13.0,3.0,1.0,14.0,3.0,4.0,3.230769
2,4.0,22.0,2.5,0.0,1.0,16.0,3.0,5.0,1.400000
3,4.0,37.0,16.5,4.0,3.0,16.0,5.0,5.0,0.727273
4,5.0,27.0,9.0,1.0,1.0,14.0,3.0,4.0,4.666666


In [15]:
# A variável dependente binária 'had_affair'(1 é sim e 0 é não)
df_real['had_affair'] = (affair.endog > 0).astype(int)
df_real = df_real[['had_affair','rate_marriage','age','educ']]
display(df_real.describe())

,had_affair,rate_marriage,age,educ
count,6366.000000,6366.000000,6366.000000,6366.000000
mean,0.322495,4.109645,29.082862,14.209865
std,0.467468,0.961430,6.847882,2.178003
min,0.000000,1.000000,17.500000,9.000000
25%,0.000000,4.000000,22.000000,12.000000
50%,0.000000,4.000000,27.000000,14.000000
75%,1.000000,5.000000,32.000000,16.000000
max,1.000000,5.000000,42.000000,20.000000


## Amostra de teste e treino

In [5]:
num_splits = 4
kf = KFold(n_splits=num_splits, shuffle=True, random_state=42)
kf

KFold(n_splits=4, random_state=42, shuffle=True)

In [16]:
kf_iterator = iter(kf.split(df_real))
train_index_0, test_index_0 = next(kf_iterator)
print(f"Tamanho do conjunto de treino: {len(train_index_0)}")
print(f"Tamanho do conjunto de teste: {len(test_index_0)}")
display(df_real.iloc[train_index_0].describe())
display(df_real.iloc[test_index_0].describe())

Tamanho do conjunto de treino: 4774
Tamanho do conjunto de teste: 1592


,had_affair,rate_marriage,age,educ
count,4774.000000,4774.000000,4774.000000,4774.000000
mean,0.317553,4.117721,29.007122,14.224340
std,0.465573,0.959241,6.833811,2.168027
min,0.000000,1.000000,17.500000,9.000000
25%,0.000000,4.000000,22.000000,12.000000
50%,0.000000,4.000000,27.000000,14.000000
75%,1.000000,5.000000,32.000000,16.000000
max,1.000000,5.000000,42.000000,20.000000


,had_affair,rate_marriage,age,educ
count,1592.000000,1592.000000,1592.000000,1592.000000
mean,0.337312,4.085427,29.309987,14.166457
std,0.472941,0.967863,6.887065,2.207764
min,0.000000,1.000000,17.500000,9.000000
25%,0.000000,4.000000,22.000000,12.000000
50%,0.000000,4.000000,27.000000,14.000000
75%,1.000000,5.000000,32.000000,16.000000
max,1.000000,5.000000,42.000000,20.000000


In [17]:
class RegressionModel:
    def __init__(self, model_type):
        self.model_type = model_type
        self.model = None

    def fit(self, formula, data):
        if self.model_type == 'LPM':
            self.model = smf.ols(formula, data=data).fit()
        elif self.model_type == 'Logit':
            self.model = smf.logit(formula, data=data).fit()
        elif self.model_type == 'Probit':
            self.model = smf.probit(formula, data=data).fit()
        else:
            raise ValueError("Tipo de modelo inválido. Escolha 'LPM', 'Logit' ou 'Probit'.")

    def predict(self, data):
        if self.model:
            return self.model.predict(data)
        else:
            raise ValueError("O modelo não foi ajustado ainda. Chame .fit() primeiro.")


print("Classe RegressionModel e função de métricas definidas.")

Classe RegressionModel e função de métricas definidas.


In [18]:
# Definindo a fórmula do modelo
formula = 'had_affair ~ rate_marriage + age + educ'
model =  'LPM'

# Lista para armazenar os resultados de cada fold
all_fold_metrics = []

for i, (train_index, test_index) in enumerate(kf.split(df_real)):
    print(f"\n--- Amostra de Treino/Teste {i+1}/{num_splits} ---")

    # Dividir os dados em treino e teste (usando KFold para splits não sobrepostos)
    df_train = df_real.iloc[train_index]
    df_test = df_real.iloc[test_index]

    # Instanciar e ajustar os modelos
    classifier = RegressionModel(model)
    classifier.fit(formula, df_train)


    # Fazer previsões no conjunto de teste
    y_true = df_test['had_affair']

    preds = classifier.predict(df_test)

    # Converter previsões para classes binárias (0 ou 1) usando um threshold de 0.5
    preds_class = (preds > 0.5).astype(int)

    # Avaliar o modelo e armazenar as métricas para o fold atual
    fold_metrics = {
        'Fold': i + 1,
        'Model': model,
        'Accuracy': accuracy_score(y_true, preds_class),
        'Precision': precision_score(y_true, preds_class, zero_division=0),
        'Recall': recall_score(y_true, preds_class, zero_division=0),
        'F1-Score': f1_score(y_true, preds_class, zero_division=0),
        'AUC-ROC': roc_auc_score(y_true, preds)
    }
    all_fold_metrics.append(fold_metrics)

print("\n--- Avaliação Completa ---")
# Criar um DataFrame a partir da lista de dicionários
results_df1 = pd.DataFrame(all_fold_metrics)
display(results_df1)


--- Amostra de Treino/Teste 1/4 ---

--- Amostra de Treino/Teste 2/4 ---

--- Amostra de Treino/Teste 3/4 ---

--- Amostra de Treino/Teste 4/4 ---

--- Avaliação Completa ---


,Fold,Model,Accuracy,Precision,Recall,F1-Score,AUC-ROC
0,1,LPM,0.714196,0.691589,0.275605,0.394141,0.738724
1,2,LPM,0.719221,0.612069,0.284569,0.388509,0.690087
2,3,LPM,0.691389,0.588745,0.255639,0.356488,0.715321
3,4,LPM,0.723444,0.600000,0.278351,0.380282,0.712614


In [19]:
df_mean = results_df1[['Accuracy','Precision','Recall','F1-Score','AUC-ROC']].mean()
df_mean

,0
Accuracy,0.712063
Precision,0.623101
Recall,0.273541
F1-Score,0.379855
AUC-ROC,0.714187


In [20]:
# Definindo a fórmula do modelo
formula = 'had_affair ~ rate_marriage + age + educ'
model =  'Logit'

# Lista para armazenar os resultados de cada fold
all_fold_metrics = []

for i, (train_index, test_index) in enumerate(kf.split(df_real)):
    print(f"\n--- Amostra de Treino/Teste {i+1}/{num_splits} ---")

    # Dividir os dados em treino e teste (usando KFold para splits não sobrepostos)
    df_train = df_real.iloc[train_index]
    df_test = df_real.iloc[test_index]

    # Instanciar e ajustar os modelos
    classifier = RegressionModel(model)
    classifier.fit(formula, df_train)


    # Fazer previsões no conjunto de teste
    y_true = df_test['had_affair']

    preds = classifier.predict(df_test)

    # Converter previsões para classes binárias (0 ou 1) usando um threshold de 0.5
    preds_class = (preds > 0.5).astype(int)

    # Avaliar o modelo e armazenar as métricas para o fold atual
    fold_metrics = {
        'Fold': i + 1,
        'Model': model,
        'Accuracy': accuracy_score(y_true, preds_class),
        'Precision': precision_score(y_true, preds_class, zero_division=0),
        'Recall': recall_score(y_true, preds_class, zero_division=0),
        'F1-Score': f1_score(y_true, preds_class, zero_division=0),
        'AUC-ROC': roc_auc_score(y_true, preds)
    }
    all_fold_metrics.append(fold_metrics)

print("\n--- Avaliação Completa ---")
# Criar um DataFrame a partir da lista de dicionários
results_df2 = pd.DataFrame(all_fold_metrics)
display(results_df2)


--- Amostra de Treino/Teste 1/4 ---
Optimization terminated successfully.
         Current function value: 0.567894
         Iterations 5

--- Amostra de Treino/Teste 2/4 ---
Optimization terminated successfully.
         Current function value: 0.564069
         Iterations 5

--- Amostra de Treino/Teste 3/4 ---
Optimization terminated successfully.
         Current function value: 0.562543
         Iterations 5

--- Amostra de Treino/Teste 4/4 ---
Optimization terminated successfully.
         Current function value: 0.569325
         Iterations 5

--- Avaliação Completa ---


,Fold,Model,Accuracy,Precision,Recall,F1-Score,AUC-ROC
0,1,Logit,0.714824,0.691244,0.279330,0.397878,0.738733
1,2,Logit,0.719221,0.611111,0.286573,0.390177,0.690966
2,3,Logit,0.691389,0.588745,0.255639,0.356488,0.714359
3,4,Logit,0.722816,0.595652,0.282474,0.383217,0.711261


In [21]:
# Definindo a fórmula do modelo
formula = 'had_affair ~ rate_marriage + age + educ'
model =  'Probit'

# Lista para armazenar os resultados de cada fold
all_fold_metrics = []

for i, (train_index, test_index) in enumerate(kf.split(df_real)):
    print(f"\n--- Amostra de Treino/Teste {i+1}/{num_splits} ---")

    # Dividir os dados em treino e teste (usando KFold para splits não sobrepostos)
    df_train = df_real.iloc[train_index]
    df_test = df_real.iloc[test_index]

    # Instanciar e ajustar os modelos
    classifier = RegressionModel(model)
    classifier.fit(formula, df_train)


    # Fazer previsões no conjunto de teste
    y_true = df_test['had_affair']

    preds = classifier.predict(df_test)

    # Converter previsões para classes binárias (0 ou 1) usando um threshold de 0.5
    preds_class = (preds > 0.5).astype(int)

    # Avaliar o modelo e armazenar as métricas para o fold atual
    fold_metrics = {
        'Fold': i + 1,
        'Model': model,
        'Accuracy': accuracy_score(y_true, preds_class),
        'Precision': precision_score(y_true, preds_class, zero_division=0),
        'Recall': recall_score(y_true, preds_class, zero_division=0),
        'F1-Score': f1_score(y_true, preds_class, zero_division=0),
        'AUC-ROC': roc_auc_score(y_true, preds)
    }
    all_fold_metrics.append(fold_metrics)

print("\n--- Avaliação Completa ---")
# Criar um DataFrame a partir da lista de dicionários
results_df3 = pd.DataFrame(all_fold_metrics)
display(results_df3)


--- Amostra de Treino/Teste 1/4 ---
Optimization terminated successfully.
         Current function value: 0.567709
         Iterations 5

--- Amostra de Treino/Teste 2/4 ---
Optimization terminated successfully.
         Current function value: 0.563812
         Iterations 5

--- Amostra de Treino/Teste 3/4 ---
Optimization terminated successfully.
         Current function value: 0.562356
         Iterations 5

--- Amostra de Treino/Teste 4/4 ---
Optimization terminated successfully.
         Current function value: 0.569123
         Iterations 5

--- Avaliação Completa ---


,Fold,Model,Accuracy,Precision,Recall,F1-Score,AUC-ROC
0,1,Probit,0.714196,0.691589,0.275605,0.394141,0.738761
1,2,Probit,0.718593,0.609442,0.284569,0.387978,0.691015
2,3,Probit,0.691389,0.588745,0.255639,0.356488,0.714402
3,4,Probit,0.722816,0.595652,0.282474,0.383217,0.711028


In [22]:
df_results = pd.concat([results_df1, results_df2, results_df3])
display(df_results)

,Fold,Model,Accuracy,Precision,Recall,F1-Score,AUC-ROC
0,1,LPM,0.714196,0.691589,0.275605,0.394141,0.738724
1,2,LPM,0.719221,0.612069,0.284569,0.388509,0.690087
2,3,LPM,0.691389,0.588745,0.255639,0.356488,0.715321
3,4,LPM,0.723444,0.600000,0.278351,0.380282,0.712614
0,1,Logit,0.714824,0.691244,0.279330,0.397878,0.738733
1,2,Logit,0.719221,0.611111,0.286573,0.390177,0.690966
2,3,Logit,0.691389,0.588745,0.255639,0.356488,0.714359
3,4,Logit,0.722816,0.595652,0.282474,0.383217,0.711261
0,1,Probit,0.714196,0.691589,0.275605,0.394141,0.738761
1,2,Probit,0.718593,0.609442,0.284569,0.387978,0.691015


In [26]:
df_results.groupby('Model').mean()

,Fold,Accuracy,Precision,Recall,F1-Score,AUC-ROC
Model,,,,,,
LPM,2.5,0.712063,0.623101,0.273541,0.379855,0.714187
Logit,2.5,0.712063,0.621688,0.276004,0.381940,0.713830
Probit,2.5,0.711748,0.621357,0.274572,0.380456,0.713802


# Atividade

## 1. Implemente o exercício fora da amostra para prever os preços dos imóveis da California?

### 1.1 - Abra base de dados:
##### housing = fetch_california_housing()
##### df_real = pd.DataFrame(housing.data, columns=housing.feature_names)
##### df_real['MEDV'] = housing.target

### 1.2 - Calcule o MSE, MAE, MAPE e MPE das amostras de testes

### 1.3 - Utilize 5 amostras de teste

### 1.4 - Teste os modelo lineares:

#### a) 'MEDV ~ MedInc + HouseAge + AveRooms'

#### b) 'MEDV ~ AveBedrms + Population + AveOccup'

#### c) 'MEDV ~ MedInc + HouseAge + AveRooms + AveBedrms + Population + AveOccup'

### 1.5 - Qual modelo você escolheria? Justifique.



In [57]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

In [58]:
class RegressionModel:
    def __init__(self, model_type):
        self.model_type = model_type
        self.model = None

    def fit(self, formula, data):
        if self.model_type == 'LPM':
            self.model = smf.ols(formula, data=data).fit()
        elif self.model_type == 'Logit':
            self.model = smf.logit(formula, data=data).fit()
        elif self.model_type == 'Probit':
            self.model = smf.probit(formula, data=data).fit()
        else:
            raise ValueError("Tipo de modelo inválido. Escolha 'LPM', 'Logit' ou 'Probit'.")

    def predict(self, data):
        if self.model:
            return self.model.predict(data)
        else:
            raise ValueError("O modelo não foi ajustado ainda. Chame .fit() primeiro.")


print("Classe RegressionModel e função de métricas definidas.")

Classe RegressionModel e função de métricas definidas.


In [59]:
housing = fetch_california_housing()
df_real = pd.DataFrame(housing.data, columns=housing.feature_names)
df_real['MEDV'] = housing.target
display(df_real.head())

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MEDV
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


In [60]:
num_splits = 5
kf = KFold(n_splits=num_splits, shuffle=True, random_state=42)

In [61]:
# Definindo a fórmula do modelo
formula = 'MEDV ~ MedInc + HouseAge + AveRooms'
model =  'm1'

# Lista para armazenar os resultados de cada fold
all_fold_metrics = []

for i, (train_index, test_index) in enumerate(kf.split(df_real)): # kf.split deve usar df_real que contem affair_scaled
    print(f"\n--- Amostra de Treino/Teste {i+1}/{num_splits} ---")

    # Dividir os dados em treino e teste (usando KFold para splits não sobrepostos)
    df_train = df_real.iloc[train_index]
    df_test = df_real.iloc[test_index]

    # Instanciar e ajustar os modelos
    classifier = RegressionModel('LPM')
    classifier.fit(formula, df_train)


    # Fazer previsões no conjunto de teste
    y_true_regression = df_test['MEDV']

    # As previsões são contínuas para métricas de regressão
    preds_regression = classifier.predict(df_test)

    # Avaliar o modelo e armazenar as métricas para o fold atual
    fold_metrics = {
        'Fold': i + 1,
        'Model': model,
        'MSE': mean_squared_error(y_true_regression, preds_regression),
        'MAE': mean_absolute_error(y_true_regression, preds_regression),
        'MAPE': mean_absolute_percentage_error(y_true_regression, preds_regression),
        'MPE': np.mean(((y_true_regression - preds_regression) / (y_true_regression )))
    }
    all_fold_metrics.append(fold_metrics)

print("\n--- Avaliação Completa ---")
results_df1 = pd.DataFrame(all_fold_metrics)
display(results_df1)


--- Amostra de Treino/Teste 1/5 ---

--- Amostra de Treino/Teste 2/5 ---

--- Amostra de Treino/Teste 3/5 ---

--- Amostra de Treino/Teste 4/5 ---

--- Amostra de Treino/Teste 5/5 ---

--- Avaliação Completa ---


,Fold,Model,MSE,MAE,MAPE,MPE
0,1,m1,0.658911,0.603321,0.377121,-0.184642
1,2,m1,0.651674,0.603701,0.372051,-0.178308
2,3,m1,0.637038,0.596838,0.371290,-0.185796
3,4,m1,0.621985,0.592298,0.374958,-0.188812
4,5,m1,0.687255,0.604420,0.367716,-0.174156


In [62]:
# Definindo a fórmula do modelo
formula = 'MEDV ~ AveBedrms + Population + AveOccup'
model =  'm2'

# Lista para armazenar os resultados de cada fold
all_fold_metrics = []

for i, (train_index, test_index) in enumerate(kf.split(df_real)): # kf.split deve usar df_real que contem affair_scaled
    print(f"\n--- Amostra de Treino/Teste {i+1}/{num_splits} ---")

    # Dividir os dados em treino e teste (usando KFold para splits não sobrepostos)
    df_train = df_real.iloc[train_index]
    df_test = df_real.iloc[test_index]

    # Instanciar e ajustar os modelos
    classifier = RegressionModel('LPM')
    classifier.fit(formula, df_train)


    # Fazer previsões no conjunto de teste
    y_true_regression = df_test['MEDV']

    # As previsões são contínuas para métricas de regressão
    preds_regression = classifier.predict(df_test)

    # Avaliar o modelo e armazenar as métricas para o fold atual
    fold_metrics = {
        'Fold': i + 1,
        'Model': model,
        'MSE': mean_squared_error(y_true_regression, preds_regression),
        'MAE': mean_absolute_error(y_true_regression, preds_regression),
        'MAPE': mean_absolute_percentage_error(y_true_regression, preds_regression),
        'MPE': np.mean(((y_true_regression - preds_regression) / (y_true_regression )))
    }
    all_fold_metrics.append(fold_metrics)

print("\n--- Avaliação Completa ---")
results_df2 = pd.DataFrame(all_fold_metrics)
display(results_df2)


--- Amostra de Treino/Teste 1/5 ---

--- Amostra de Treino/Teste 2/5 ---

--- Amostra de Treino/Teste 3/5 ---

--- Amostra de Treino/Teste 4/5 ---

--- Amostra de Treino/Teste 5/5 ---

--- Avaliação Completa ---


,Fold,Model,MSE,MAE,MAPE,MPE
0,1,m2,1.309580,0.904586,0.628905,-0.390380
1,2,m2,1.360679,0.920204,0.613218,-0.357502
2,3,m2,1.294772,0.900528,0.607383,-0.369244
3,4,m2,1.358282,0.913026,0.634617,-0.388011
4,5,m2,1.347351,0.910183,0.617060,-0.373517


In [63]:
# Definindo a fórmula do modelo
formula = 'MEDV ~ MedInc + HouseAge + AveRooms + AveBedrms + Population + AveOccup'
model =  'm3'

# Lista para armazenar os resultados de cada fold
all_fold_metrics = []

for i, (train_index, test_index) in enumerate(kf.split(df_real)): # kf.split deve usar df_real que contem affair_scaled
    print(f"\n--- Amostra de Treino/Teste {i+1}/{num_splits} ---")

    # Dividir os dados em treino e teste (usando KFold para splits não sobrepostos)
    df_train = df_real.iloc[train_index]
    df_test = df_real.iloc[test_index]

    # Instanciar e ajustar os modelos
    classifier = RegressionModel('LPM')
    classifier.fit(formula, df_train)


    # Fazer previsões no conjunto de teste
    y_true_regression = df_test['MEDV']

    # As previsões são contínuas para métricas de regressão
    preds_regression = classifier.predict(df_test)

    # Avaliar o modelo e armazenar as métricas para o fold atual
    fold_metrics = {
        'Fold': i + 1,
        'Model': model,
        'MSE': mean_squared_error(y_true_regression, preds_regression),
        'MAE': mean_absolute_error(y_true_regression, preds_regression),
        'MAPE': mean_absolute_percentage_error(y_true_regression, preds_regression),
        'MPE': np.mean(((y_true_regression - preds_regression) / (y_true_regression )))
    }
    all_fold_metrics.append(fold_metrics)

print("\n--- Avaliação Completa ---")
results_df3 = pd.DataFrame(all_fold_metrics)
display(results_df3)


--- Amostra de Treino/Teste 1/5 ---

--- Amostra de Treino/Teste 2/5 ---

--- Amostra de Treino/Teste 3/5 ---

--- Amostra de Treino/Teste 4/5 ---

--- Amostra de Treino/Teste 5/5 ---

--- Avaliação Completa ---


,Fold,Model,MSE,MAE,MAPE,MPE
0,1,m3,0.642187,0.579214,0.357330,-0.172891
1,2,m3,0.614930,0.576242,0.351726,-0.168767
2,3,m3,0.593145,0.569922,0.348955,-0.170454
3,4,m3,0.580602,0.565412,0.353870,-0.174672
4,5,m3,0.665596,0.580447,0.350589,-0.166968


In [64]:
df_results = pd.concat([results_df1, results_df2, results_df3])
df_results.groupby('Model').mean()

,Fold,MSE,MAE,MAPE,MPE
Model,,,,,
m1,3.0,0.651373,0.600116,0.372627,-0.182343
m2,3.0,1.334133,0.909705,0.620237,-0.375731
m3,3.0,0.619292,0.574248,0.352494,-0.170751
